# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the same cohort, gate and model as ML-08 and ML-09

Nothing new is fitted here. The playbook has to describe the model that was actually validated, so
this rebuilds it verbatim: the D1 cohort, ML-07's frozen gate, and ridge on the four `LEAN` features
that survived ML-09's ablation. If any number below disagrees with ML-08 or ML-09, the setup has
drifted and the playbook is describing something else.

**One thing does change, and FlyRank asked for it.** K is no longer fixed at 100. It is a
**configurable per-client budget**, so section 1 sweeps it rather than asserting it, and reports
**macro** client metrics — the mean of per-client rates — beside the pooled figures this project
has used until now.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scipy scikit-learn python-dotenv

import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

SEED = 8
token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
daily_files = [hf_hub_download(repo_id=REPO, repo_type="dataset",
               filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
               for m in MONTHS]
dim_file = hf_hub_download(repo_id=REPO, repo_type="dataset",
                           filename="dim_content.parquet", token=token)
con = duckdb.connect()
REL = "read_parquet([" + ", ".join(f"'{f}'" for f in daily_files) + "])"
D1 = "2026-03-31"


def build_cohort(dstr, window_days=30):
    """ML-08's cohort at an arbitrary decision date and outcome window.

    window_days is a parameter because FlyRank asked for the outcome window to be
    swept rather than pinned to the queue cadence; it defaults to the 30 days every
    earlier notebook used, so the default path reproduces them exactly.
    """
    q = f"""
    WITH prior AS (
      SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS impr_90d,
        SUM(gsc_clicks) AS clicks_90d,
        SUM(gsc_sum_position) AS sum_position_90d,
        SUM(gsc_impressions) FILTER (
            WHERE report_date < DATE '{dstr}' - INTERVAL 30 DAY) AS older60_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{dstr}' - INTERVAL 30 DAY) AS recent30_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{dstr}' - INTERVAL 60 DAY
              AND report_date <  DATE '{dstr}' - INTERVAL 30 DAY) AS base30_impr
      FROM {REL}
      WHERE report_date >= DATE '{dstr}' - INTERVAL 90 DAY AND report_date < DATE '{dstr}'
      GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
    fut AS (
      SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
      WHERE report_date >= DATE '{dstr}'
        AND report_date < DATE '{dstr}' + INTERVAL {int(window_days)} DAY
      GROUP BY content_hash_id)
    SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
    FROM prior p LEFT JOIN fut f USING (content_hash_id)
    ORDER BY p.content_hash_id"""
    x = con.sql(q).df()
    for c in ["older60_impr", "recent30_impr", "base30_impr"]:
        x[c] = x[c].fillna(0)
    dd = pd.Timestamp(dstr)
    x = x.merge(con.sql(f"SELECT content_hash_id, content_created_date FROM "
                        f"read_parquet('{dim_file}')").df(), on="content_hash_id", how="left")
    x["baseline_daily"] = x["older60_impr"] / 60
    x["recent_daily"] = x["recent30_impr"] / 30
    x["future_daily"] = x["future_impr"] / window_days
    x["target"] = np.arcsinh(x["future_daily"]) - np.arcsinh(x["baseline_daily"])
    x["declined"] = x["target"] < 0
    x["avg_position"] = x["sum_position_90d"] / x["impr_90d"].replace(0, np.nan) + 1
    x["content_age_days"] = (dd - pd.to_datetime(x["content_created_date"])).dt.days
    x["slip"] = np.where(x["baseline_daily"] > 0,
                         (x["baseline_daily"] - x["recent_daily"]) / x["baseline_daily"], np.nan)
    x["peak_ratio"] = np.where(x["impr_90d"] > 0,
                               x["recent_daily"] / (x["impr_90d"] / 90), np.nan)
    x["prior_trend"] = np.where(x["base30_impr"] > 0,
                                (x["recent30_impr"] - x["base30_impr"]) / x["base30_impr"], np.nan)
    med = x.groupby("client_hash_id")["impr_90d"].transform("median")
    gate = ((x["baseline_daily"] > 0) & (x["content_age_days"] >= 180)
            & (x["impr_90d"] >= med) & (x["slip"].fillna(0) <= 0.5))
    x["in_gate"] = gate
    return x


SAFE = ["content_age_days", "impr_90d", "avg_position"]
FULL = SAFE + ["prior_trend", "peak_ratio"]
LEAN = [f for f in FULL if f != "content_age_days"]

cohort = build_cohort(D1)
ev = cohort[cohort["in_gate"]].dropna(subset=FULL).copy()

print(f"cohort      {len(cohort):>7,} pages | {cohort['client_hash_id'].nunique():>2} clients")
print(f"eval pool   {len(ev):>7,} pages | {ev['client_hash_id'].nunique():>2} clients "
      f"| decline rate {ev['declined'].mean():.4f}")
print("ML-08 reported: 45,095 pages, 30 clients, 0.5131  <- must match")
print(f"features: {LEAN}")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cohort      202,073 pages | 53 clients
eval pool    45,095 pages | 30 clients | decline rate 0.5131
ML-08 reported: 45,095 pages, 30 clients, 0.5131  <- must match
features: ['impr_90d', 'avg_position', 'prior_trend', 'peak_ratio']


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The deliverable is one ranked list per client, each row carrying the reason it is there. Two questions
have to be answered before the list means anything, and FlyRank's answer settled how to ask both.

**How deep does the list go?** K is a per-client budget, not a constant. The sweep below runs it from
10 to 500 and then to the whole pool, against a random ordering of the same gated pages at the same
depth — because ML-09 §8 established that an absolute `P@K` without its base rate is not
interpretable. **Lift over random at the same K is the number that means something.**

**Whose average?** FlyRank asks for **macro** client metrics: score each client, then average the
clients. This project has reported **pooled** (total hits over total picks) since ML-05. The two
answer different questions — pooled tells you how the whole book of work went, macro tells you how the
average client experienced it — and a client with 24,418 pages dominates the pooled figure while
counting once in the macro one. Both are reported below.

**Reason codes come from the model, not from a story about it.** Ridge on standardised features means
each page's prediction decomposes exactly: contribution of feature *j* = coefficient *j* × that page's
standardised value. The two most negative contributions are what pushed the page onto the list, and
the wording is generated from the sign of the coefficient rather than written by hand.

In [2]:
def queue_metrics(frame, score_col, k, ascending):
    """Per-client top-k. Returns pooled precision, macro precision, and clients scored.

    Pooled = total hits / total picks. Macro = mean of per-client precision, which is
    what FlyRank asks for: every client counts once regardless of size.
    """
    hits = picks = 0
    per = []
    for _, g in frame.groupby("client_hash_id"):
        kk = len(g) if k is None else min(k, len(g))
        if kk == 0:
            continue
        top = g.sort_values(score_col, ascending=ascending).head(kk)
        h = int(top["declined"].sum())
        hits += h
        picks += kk
        per.append(h / kk)
    return (hits / picks if picks else np.nan,
            float(np.mean(per)) if per else np.nan,
            len(per))


KS = [10, 25, 50, 100, 250, 500, None]          # None = the client's whole gated pool
gss = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=SEED)

rows = []
for s, (itr, ite) in enumerate(gss.split(ev, groups=ev["client_hash_id"])):
    train, test = ev.iloc[itr], ev.iloc[ite]
    sc = StandardScaler().fit(train[LEAN])
    mdl = Ridge(alpha=1.0).fit(sc.transform(train[LEAN]), train["target"])
    rng = np.random.default_rng(s)
    t = test.assign(pred=mdl.predict(sc.transform(test[LEAN])), rand=rng.random(len(test)))
    for k in KS:
        pm, mm, nc = queue_metrics(t, "pred", k, True)
        pr, mr, _ = queue_metrics(t, "rand", k, False)
        rows.append({"K": "all" if k is None else k, "seed": s,
                     "pooled": pm, "macro": mm,
                     "pooled_rand": pr, "macro_rand": mr, "clients": nc})

sw = (pd.DataFrame(rows).groupby("K", sort=False)
      .agg(pooled=("pooled", "mean"), pooled_rand=("pooled_rand", "mean"),
           macro=("macro", "mean"), macro_rand=("macro_rand", "mean"),
           clients=("clients", "mean")))
sw["pooled_lift"] = sw["pooled"] - sw["pooled_rand"]
sw["macro_lift"] = sw["macro"] - sw["macro_rand"]

print("K sweep -- ten grouped splits, per-client queues, lift is over random at the same K")
print(sw[["pooled", "pooled_rand", "pooled_lift", "macro", "macro_rand", "macro_lift"]]
      .to_string(float_format=lambda v: f"{v:.4f}"))

best_p = sw["pooled_lift"].idxmax()
best_m = sw["macro_lift"].idxmax()
print(f"\nbest pooled lift at K = {best_p} ({sw.loc[best_p, 'pooled_lift']:.4f})")
print(f"best macro  lift at K = {best_m} ({sw.loc[best_m, 'macro_lift']:.4f})")
print(f"at K = all, lift is pooled {sw.loc['all', 'pooled_lift']:+.4f} / "
      f"macro {sw.loc['all', 'macro_lift']:+.4f}  <- must be ~0: the queue is the whole pool")
print(f"\nmacro minus pooled at K = 100: {sw.loc[100, 'macro'] - sw.loc[100, 'pooled']:+.4f}"
      f"  (the average client is not the average page)")

K sweep -- ten grouped splits, per-client queues, lift is over random at the same K
     pooled  pooled_rand  pooled_lift  macro  macro_rand  macro_lift
K                                                                   
10   0.8766       0.5047       0.3719 0.8683      0.5200      0.3483
25   0.8389       0.5139       0.3250 0.8259      0.5286      0.2973
50   0.8157       0.5262       0.2895 0.7986      0.5383      0.2603
100  0.8100       0.5344       0.2755 0.7775      0.5419      0.2357
250  0.8127       0.5459       0.2667 0.7463      0.5399      0.2064
500  0.8055       0.5563       0.2492 0.7176      0.5407      0.1769
all  0.5013       0.5013       0.0000 0.5407      0.5407      0.0000

best pooled lift at K = 10 (0.3719)
best macro  lift at K = 10 (0.3483)
at K = all, lift is pooled +0.0000 / macro +0.0000  <- must be ~0: the queue is the whole pool

macro minus pooled at K = 100: -0.0324  (the average client is not the average page)


**Verdict: the model's advantage lives at the top of the list, and a single global K is the wrong
instrument — which is what makes FlyRank's "configurable per-client budget" the right one.**

**Lift falls monotonically as the queue gets deeper.** Pooled **0.3719** at K = 10 down to **0.2492**
at K = 500; macro **0.3483** down to **0.1769**. Nothing surprising in the direction — a ranker should
be most right about its top picks — but the size matters operationally: the first ten pages per client
are worth roughly half again as much per page as the five-hundredth.

**The sanity check passes exactly.** At `K = all` the queue *is* the client's whole gated pool, so
there is no ordering left to be right about and lift must be zero. It is **0.0000** on both
conventions. A metric that failed here would have been measuring the gate, not the ranking.

**Macro sits below pooled at every depth, and the gap widens with K** — **−0.0324** at K = 100. This
is not noise, it has a mechanism, and it is the finding that decides the design. Large clients have
deep pools, so a queue of 500 still selects their top 500 out of thousands. Small clients run out of
pages: their "top 500" is everything they have, no ranking occurs, and their precision collapses to
their own base rate. Pooled averaging hides those clients behind the big ones; **macro counts them
once each, so macro is the convention that can see the problem** — which is presumably why FlyRank
asks for it.

**The random bar moves with K too**, from **0.5047** at K = 10 to **0.5563** at K = 500 pooled, against
a whole-pool base rate of **0.5013**. Deepening the queue changes *which pages* are in it — small
clients contribute all of theirs — so the comparison pool is not fixed. This is why every lift above is
measured against random **at the same K**; a fixed base rate would have quietly mis-stated all of them.

Two consequences carried into the rest of this playbook: **K should be set from each client's own pool
size**, not chosen once for everyone, and **the macro figure is the one to report to a client**, because
it is the one that describes their experience rather than the book's.

In [3]:
# The deployment model: fitted on the whole evaluation pool, since there is no
# held-out set to protect at scoring time. Every performance figure quoted for it
# comes from the grouped splits above, never from this fit.
scaler = StandardScaler().fit(ev[LEAN])
model = Ridge(alpha=1.0).fit(scaler.transform(ev[LEAN]), ev["target"])

print("deployment model, ridge on standardised features:")
for f, c in zip(LEAN, model.coef_):
    print(f"  {f:<14} {c:+.4f}   (a HIGH value pushes the prediction "
          f"{'DOWN -> more likely to decline' if c < 0 else 'UP -> less likely to decline'})")

# Per-page decomposition. For a linear model on standardised inputs the prediction
# is exactly the sum of coefficient x standardised value, so the reasons a page is
# ranked where it is can be read off rather than guessed at.
Z = scaler.transform(ev[LEAN])
contrib = Z * model.coef_

PHRASE = {
    ("impr_90d", "high"): "high search volume over the last 90 days",
    ("impr_90d", "low"): "little search volume left to lose",
    ("avg_position", "high"): "ranks poorly on average",
    ("avg_position", "low"): "ranks well on average",
    ("prior_trend", "high"): "impressions jumped in the last 30 days versus the 30 before",
    ("prior_trend", "low"): "impressions already falling over the last 30 days",
    ("peak_ratio", "high"): "running above its own 90-day average",
    ("peak_ratio", "low"): "running below its own 90-day average",
}

order = np.argsort(contrib, axis=1)          # most negative contribution first
r1 = [PHRASE[(LEAN[j], "high" if Z[i, j] > 0 else "low")] for i, j in enumerate(order[:, 0])]
r2 = [PHRASE[(LEAN[j], "high" if Z[i, j] > 0 else "low")] for i, j in enumerate(order[:, 1])]

queue = ev[["client_hash_id", "content_hash_id", "impr_90d", "avg_position",
            "prior_trend", "peak_ratio", "declined"]].copy()
queue["predicted_change"] = model.predict(Z)
queue["reason_1"] = r1
queue["reason_2"] = r2
queue["rank_in_client"] = (queue.groupby("client_hash_id")["predicted_change"]
                           .rank(method="first", ascending=True).astype(int))
queue = queue.sort_values(["client_hash_id", "rank_in_client"])

print(f"\nqueue built: {len(queue):,} rows | {queue['client_hash_id'].nunique()} clients")
print("\nwhy pages reach the top of a queue (primary reason, top 100 per client):")
top = queue[queue["rank_in_client"] <= 100]
share = top["reason_1"].value_counts(normalize=True)
for reason, pct in share.items():
    print(f"  {pct:6.1%}  {reason}")

# How many clients cannot be ranked at a given budget, because the budget exceeds
# their whole pool. This is the mechanism behind the macro/pooled gap above.
print("\nclients whose entire gated pool fits inside the budget (so no ranking happens):")
sizes = queue.groupby("client_hash_id").size()
for k in [10, 25, 50, 100, 250, 500]:
    n = int((sizes <= k).sum())
    print(f"  K = {k:>3}: {n:>2} of {len(sizes)} clients ({n / len(sizes):5.1%}) "
          f"| pages covered {int(np.minimum(sizes, k).sum()):>6,} of {int(sizes.sum()):,}")

print(f"\npool size per client: min {sizes.min()}, median {int(sizes.median())}, "
      f"max {sizes.max():,}")
print(f"a budget of 10% of each client's pool would range "
      f"{max(1, int(sizes.min() * 0.1))} to {int(sizes.max() * 0.1):,} pages")

deployment model, ridge on standardised features:
  impr_90d       +0.0586   (a HIGH value pushes the prediction UP -> less likely to decline)
  avg_position   +0.0004   (a HIGH value pushes the prediction UP -> less likely to decline)
  prior_trend    +0.0296   (a HIGH value pushes the prediction UP -> less likely to decline)
  peak_ratio     +0.5149   (a HIGH value pushes the prediction UP -> less likely to decline)



queue built: 45,095 rows | 30 clients

why pages reach the top of a queue (primary reason, top 100 per client):
   82.3%  running below its own 90-day average
   16.9%  little search volume left to lose
    0.7%  impressions already falling over the last 30 days
    0.2%  ranks well on average

clients whose entire gated pool fits inside the budget (so no ranking happens):
  K =  10:  5 of 30 clients (16.7%) | pages covered    265 of 45,095
  K =  25:  8 of 30 clients (26.7%) | pages covered    616 of 45,095
  K =  50:  8 of 30 clients (26.7%) | pages covered  1,166 of 45,095
  K = 100: 13 of 30 clients (43.3%) | pages covered  2,147 of 45,095
  K = 250: 14 of 30 clients (46.7%) | pages covered  4,569 of 45,095
  K = 500: 17 of 30 clients (56.7%) | pages covered  8,246 of 45,095

pool size per client: min 1, median 353, max 8,770
a budget of 10% of each client's pool would range 1 to 877 pages


**Verdict: the queue works, and reading its own reasons back is the most uncomfortable result in this
notebook. The model is one feature, and that feature describes the present rather than forecasting the
future.**

**Every coefficient is positive, so pages reach the top of the queue by being *low* on something.**

| feature | coefficient | share of top-100 pages it explains |
|---|---|---|
| `peak_ratio` | **+0.5149** | **82.3%** — "running below its own 90-day average" |
| `impr_90d` | +0.0586 | 16.9% — "little search volume left to lose" |
| `prior_trend` | +0.0296 | 0.7% — "impressions already falling" |
| `avg_position` | **+0.0004** | 0.2% — "ranks well on average" |

**This is a one-feature model wearing four.** On standardised inputs the coefficients are directly
comparable, and `peak_ratio` is nearly nine times `impr_90d` and more than a thousand times
`avg_position`. Ridge kept `avg_position` because dropping it costs nothing, not because it does
anything — at **+0.0004** it is inert in this fit. Anything the playbook says about *why* a page is
queued is, four times in five, a statement about `peak_ratio`.

**Deriving the wording from the coefficient rather than from memory was not a formality.** The obvious
reason text — *"this page is at a peak, so it has further to fall"* — is the mean-reversion story this
project told for weeks, and under the current baseline it is **backwards**. `peak_ratio` correlated
**−0.15** with the target under the superseded baseline and **+0.51** under this one, and the sign that
survived is the one saying pages already *below* their average keep falling. Hand-written reason codes
would have shipped the inverse of the model's actual logic to a client.

**And that is the question section 2 has to settle.** "Running below its own 90-day average"
describes what already happened. If the model can only *order* pages that have already started
falling, the queue is a triage tool and must not be sold as early warning. **Composition alone cannot
answer that** — a queue can be full of already-falling pages while the model still ranks the rest
perfectly well. Section 2 tests it directly, and the answer is not the one this paragraph first
assumed.

**The budget cannot be one number.** Pools run from **1** page to **8,770**, median **353**, and a fixed
K stops ranking for more and more clients as it grows: **13 of 30** clients receive their entire
eligible list at K = 100, **17 of 30** at K = 500. Those clients are handed the gate, not a ranking —
ML-07 already found the gate carries information while the ordering inside it did not, so this is the
quantified version of ML-09 §2a's warning. **A budget set as a share of each client's pool ranks every
client by construction**; at 10% it would run 1 to 877 pages. The exact fraction is a capacity decision,
not a modelling one.

**Coverage is now a choice rather than a ceiling.** A K = 100 budget touches **2,147** of **45,095**
gated pages; K = 500 touches **8,246**. The withdrawn "4.2% of declines is the ceiling" claim came from
treating 100 as fixed — with K configurable, coverage is bought rather than capped.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** A senior SEO specialist, once per weekly rebuild, deciding which of their client's
pages to open first. The output is an ordering plus the reason for it. It is decision support: the
model never edits a page, never files a ticket, and its ranking is an argument a specialist can
overrule.

**Not intended for**, and none of these is supported by anything measured in this project: estimating
how much traffic a page will lose, deciding whether a page is worth keeping, comparing one client's
performance against another's, or claiming that acting on a queued page caused what happened next.
ML-08 measured **R² 0.0026** — the magnitude is not predictable, only the ordering.

Two limits are tested below rather than asserted, because both decide whether the tool fits the job.

**Does the outcome window matter?** Every number in this project labels a page by what happened in the
**30 days** after the decision point, and that 30 was chosen to match a monthly queue rebuild. FlyRank
rejected that coupling, so the sweep below relabels the *same pages, same features, same model* at 7,
14, 30, 60 and 90 days and asks whether the ranking still works.

**Is this early warning or triage?** Section 1 found 82.3% of top-of-queue pages are there because they
are already running below their own 90-day average — a description of the past. If the model can only
order pages that have *already* started falling, it is a triage tool and must not be sold as an early
warning. Splitting the pool on that exact condition tests it directly.

In [4]:
WINDOWS = [7, 14, 30, 60, 90]

# One pass over the future window rather than one query per horizon: the prior-side
# aggregation is identical for every window, so only the future sums need re-cutting.
qf = f"""SELECT content_hash_id,
  {', '.join(f"SUM(gsc_impressions) FILTER (WHERE report_date < DATE '{D1}' + INTERVAL {w} DAY)"
             f" AS fut_{w}" for w in WINDOWS)}
FROM {REL}
WHERE report_date >= DATE '{D1}' AND report_date < DATE '{D1}' + INTERVAL {max(WINDOWS)} DAY
GROUP BY content_hash_id ORDER BY content_hash_id"""
fut = con.sql(qf).df()
wv = ev.merge(fut, on="content_hash_id", how="left")
for w in WINDOWS:
    wv[f"fut_{w}"] = wv[f"fut_{w}"].fillna(0)


def lift_at(frame, target_col, k=100, n_splits=10):
    """Macro and pooled lift over random at the same K, on grouped splits."""
    frame = frame.copy()
    frame["declined"] = frame[target_col] < 0
    g = GroupShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=SEED)
    P_, M_ = [], []
    for s, (itr, ite) in enumerate(g.split(frame, groups=frame["client_hash_id"])):
        tr, te = frame.iloc[itr], frame.iloc[ite]
        sc = StandardScaler().fit(tr[LEAN])
        m = Ridge(alpha=1.0).fit(sc.transform(tr[LEAN]), tr[target_col])
        rng = np.random.default_rng(s)
        t = te.assign(pred=m.predict(sc.transform(te[LEAN])), rand=rng.random(len(te)))
        pm, mm, _ = queue_metrics(t, "pred", k, True)
        pr, mr, _ = queue_metrics(t, "rand", k, False)
        P_.append(pm - pr)
        M_.append(mm - mr)
    return float(np.mean(P_)), float(np.mean(M_)), float(frame["declined"].mean())


print("outcome-window sweep -- same pages, same features, only the label window moves")
print(f"{'window':>7} {'base rate':>10} {'pooled lift':>12} {'macro lift':>11}")
wrows = []
for w in WINDOWS:
    wv[f"target_{w}"] = np.arcsinh(wv[f"fut_{w}"] / w) - np.arcsinh(wv["baseline_daily"])
    pl, ml, br = lift_at(wv, f"target_{w}")
    wrows.append({"window": w, "base_rate": br, "pooled_lift": pl, "macro_lift": ml})
    print(f"{w:>5}d {br:>10.4f} {pl:>12.4f} {ml:>11.4f}")

ws = pd.DataFrame(wrows).set_index("window")
print(f"\npooled lift range {ws['pooled_lift'].min():.4f} - {ws['pooled_lift'].max():.4f}"
      f" | best at {ws['pooled_lift'].idxmax()}d, worst at {ws['pooled_lift'].idxmin()}d")
print(f"base rate range   {ws['base_rate'].min():.4f} - {ws['base_rate'].max():.4f}")
print(f"every window beats random: {(ws['pooled_lift'] > 0).all()}")

# ML-09 section 8b: a queue drawn from a pool declining at b cannot beat random by
# more than 1 - b, so raw lifts measured at different base rates are not comparable.
# The base rate moves across these windows, so normalise before reading the trend.
ws["headroom"] = 1 - ws["base_rate"]
ws["pct_ceiling"] = ws["pooled_lift"] / ws["headroom"]
print("\nthe same lifts as a share of the ceiling each base rate allows:")
print(ws[["base_rate", "headroom", "pooled_lift", "pct_ceiling"]]
      .to_string(float_format=lambda v: f"{v:.4f}"))
print(f"  capture {ws['pct_ceiling'].min():.1%} - {ws['pct_ceiling'].max():.1%}, "
      f"mean {ws['pct_ceiling'].mean():.1%}")

# --- early warning or triage? ---------------------------------------------------
print("\n" + "=" * 66)
print("early warning or triage: split on the condition that drives 82.3% of the queue")
print("=" * 66)
for label, mask in [("already BELOW its 90-day average (peak_ratio < 1)", wv["peak_ratio"] < 1),
                    ("at or ABOVE its 90-day average (peak_ratio >= 1)", wv["peak_ratio"] >= 1)]:
    sub = wv[mask]
    ncl = sub["client_hash_id"].nunique()
    if len(sub) < 500 or ncl < 6:
        print(f"\n{label}: {len(sub):,} pages / {ncl} clients -- too few to split, skipped")
        continue
    pl, ml, br = lift_at(sub, "target_30")
    print(f"\n{label}")
    print(f"  {len(sub):>6,} pages | {ncl} clients | decline rate {br:.4f}")
    print(f"  pooled lift {pl:+.4f} | macro lift {ml:+.4f}")
    print(f"  headroom {1 - br:.4f} -> captures {pl / (1 - br):.1%} of what is available")

outcome-window sweep -- same pages, same features, only the label window moves
 window  base rate  pooled lift  macro lift


    7d     0.4826       0.3061      0.2613


   14d     0.4774       0.3000      0.2560


   30d     0.5131       0.2755      0.2357


   60d     0.5520       0.2404      0.2052


   90d     0.6166       0.2137      0.1823

pooled lift range 0.2137 - 0.3061 | best at 7d, worst at 90d
base rate range   0.4774 - 0.6166
every window beats random: True

the same lifts as a share of the ceiling each base rate allows:
        base_rate  headroom  pooled_lift  pct_ceiling
window                                               
7          0.4826    0.5174       0.3061       0.5916
14         0.4774    0.5226       0.3000       0.5740
30         0.5131    0.4869       0.2755       0.5659
60         0.5520    0.4480       0.2404       0.5367
90         0.6166    0.3834       0.2137       0.5575
  capture 53.7% - 59.2%, mean 56.5%

early warning or triage: split on the condition that drives 82.3% of the queue



already BELOW its 90-day average (peak_ratio < 1)
  13,410 pages | 28 clients | decline rate 0.7641
  pooled lift +0.0787 | macro lift +0.0418
  headroom 0.2359 -> captures 33.4% of what is available



at or ABOVE its 90-day average (peak_ratio >= 1)
  31,685 pages | 30 clients | decline rate 0.4069
  pooled lift +0.2186 | macro lift +0.1802
  headroom 0.5931 -> captures 36.9% of what is available


**Verdict: the outcome window is free to choose, the model is not a triage tool, and the global queue
puts its effort where the model helps least.**

**The label window can be set on operational grounds.** Raw lift falls steadily as the window
lengthens — **0.3061** at 7 days to **0.2137** at 90 — which looks like decay until the base rate is
put beside it. It climbs from **0.4826** to **0.6166** over the same range, and a queue drawn from a
pool declining at rate `b` cannot beat random by more than `1 − b`.

| window | base rate | headroom | pooled lift | share of ceiling |
|---|---|---|---|---|
| 7d | 0.4826 | 0.5174 | 0.3061 | **0.5916** |
| 14d | 0.4774 | 0.5226 | 0.3000 | **0.5740** |
| 30d | 0.5131 | 0.4869 | 0.2755 | **0.5659** |
| 60d | 0.5520 | 0.4480 | 0.2404 | **0.5367** |
| 90d | 0.6166 | 0.3834 | 0.2137 | **0.5575** |

Capture runs **53.7% - 59.2%**, mean **56.5%**, with no trend. **The model is equally skilful at every
horizon from a week to a quarter** — the falling lift is the shrinking ceiling, exactly as ML-09 §8
found across months. **Every window beats random.** So the 30 days this project used is not special,
and nothing forces it to match the rebuild cadence. FlyRank's instruction to decouple them costs
nothing, and a weekly rebuild reporting a 7-day outcome is as well supported as anything else here.

**The model is not merely triaging pages that have already fallen — section 1 guessed wrong.** Raw
lift on pages already below their 90-day average is **+0.0787**, against **+0.2186** for pages at or
above it, a gap of nearly three times that looks damning. It is mostly ceiling again: those segments
decline at **0.7641** and **0.4069**, leaving headroom of **0.2359** against **0.5931**. Normalised,
the model captures **33.4%** of what is available in the already-falling segment and **36.9%** in the
not-yet-falling one. **Comparable skill, marginally better on the pages that have not yet moved.**

**But the useful conclusion is the one about where the model earns its place.** A page already below
its own average declines **76.41%** of the time — that is a one-line filter, and it needs no model at
all. The ranking adds **+0.0787** on top of it. On pages that have *not* started falling, where a
filter would leave you at **40.69%**, the ranking adds **+0.2186**. **The model is roughly three times
more valuable exactly where a simple rule is useless** — and section 1 showed the global score sends
**82.3%** of the top-of-queue slots to the segment where it helps least, because low `peak_ratio`
dominates the score.

**Recommendation: run the queue within segments, not globally.** Pages already below their average are
a filter output and should be presented as one; pages at or above it are where the ranking is worth a
specialist's time. A single merged list buries the second group under the first.

**Where this stops being valid.** All of it is measured at one decision point, 2026-03-31, on 30
clients that survived the gate; ML-09 §8 showed the base rate is not stationary across months, so
these levels will move even if the skill does not. The segment split is observational — pages were not
assigned to be above or below their average — so it describes where the model works, not why. And
nothing here licenses a claim about magnitude: ML-08 measured **R² 0.0026**, so the ordering is the
product and the predicted value is not.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A no-go list is only worth writing if it names where the model **demonstrably** stops working, so the
three candidates below are measured rather than assumed.

**Does the ordering hold all the way down?** The queue is used top-first, but a specialist with budget
left works into the middle of it. Binning out-of-fold predictions into deciles shows whether the
ordering degrades gracefully or falls off somewhere specific.

**Are small pages rankable at all?** `impr_90d` is a feature, and low-volume pages are noisier by
construction — a page going from 3 impressions to 1 is a 67% drop and means nothing. If the model
cannot order the bottom volume quartile, those pages should not be in a queue.

**How much of the queue is already dead?** ML-06 found pages reaching zero need surfacing but not
competing for queue position, because a page at zero needs a different decision from a page sliding.

In [5]:
# --- 1. does the ordering hold all the way down the list? -----------------------
drows = []
for s, (itr, ite) in enumerate(gss.split(ev, groups=ev["client_hash_id"])):
    tr, te = ev.iloc[itr], ev.iloc[ite]
    sc = StandardScaler().fit(tr[LEAN])
    m = Ridge(alpha=1.0).fit(sc.transform(tr[LEAN]), tr["target"])
    t = te.assign(pred=m.predict(sc.transform(te[LEAN])))
    t = t.assign(decile=pd.qcut(t["pred"], 10, labels=False, duplicates="drop") + 1)
    for d, g in t.groupby("decile"):
        drows.append({"decile": int(d), "rate": g["declined"].mean(), "n": len(g)})

dec = (pd.DataFrame(drows).groupby("decile")
       .agg(decline_rate=("rate", "mean"), pages=("n", "mean")))
pool_rate = ev["declined"].mean()
dec["vs_pool"] = dec["decline_rate"] - pool_rate

print("actual decline rate by predicted decile (1 = model says most likely to fall)")
print(f"pool decline rate {pool_rate:.4f}")
print(dec.to_string(float_format=lambda v: f"{v:.4f}"))
mono = bool((dec["decline_rate"].diff().dropna() < 0).all())
print(f"\nmonotonic across all ten deciles: {mono}")
print(f"top decile {dec.loc[1, 'decline_rate']:.4f} | "
      f"bottom decile {dec.loc[dec.index.max(), 'decline_rate']:.4f} | "
      f"spread {dec.loc[1, 'decline_rate'] - dec.loc[dec.index.max(), 'decline_rate']:.4f}")
crossover = dec.index[dec["vs_pool"] <= 0].min()
print(f"first decile no better than the pool base rate: {crossover}")

# --- 2. are low-volume pages rankable? ------------------------------------------
print("\n" + "=" * 70)
print("does the ranking work at every traffic volume?")
print("=" * 70)
ev_v = ev.assign(vol_q=pd.qcut(ev["impr_90d"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"]))
print(f"{'quartile':<12} {'pages':>7} {'impr_90d range':>22} {'base':>7} {'lift':>8} {'ceiling':>9}")
for q, g in ev_v.groupby("vol_q", observed=True):
    if g["client_hash_id"].nunique() < 6:
        print(f"{str(q):<12} {len(g):>7,}  too few clients to split")
        continue
    pl, ml, br = lift_at(g, "target")
    print(f"{str(q):<12} {len(g):>7,} {int(g['impr_90d'].min()):>9,} - {int(g['impr_90d'].max()):>9,}"
          f" {br:>7.4f} {pl:>+8.4f} {pl / (1 - br):>8.1%}")

# --- 3. how much of the queue is already dead? ----------------------------------
print("\n" + "=" * 70)
print("pages already at zero: a different decision, not a queue position")
print("=" * 70)
ev_d = ev.assign(dead=ev["future_impr"] == 0)
top100 = (ev_d.assign(pred=model.predict(scaler.transform(ev_d[LEAN])))
          .sort_values("pred").groupby("client_hash_id").head(100))
print(f"gated pool reaching zero in the outcome window: {ev_d['dead'].sum():,} "
      f"of {len(ev_d):,} ({ev_d['dead'].mean():.2%})")
print(f"of the top 100 per client:                      {int(top100['dead'].sum()):,} "
      f"of {len(top100):,} ({top100['dead'].mean():.2%})")
print(f"median impr_90d of a dead queued page: {int(top100.loc[top100['dead'], 'impr_90d'].median()):,}"
      f" | of a live one: {int(top100.loc[~top100['dead'], 'impr_90d'].median()):,}")

actual decline rate by predicted decile (1 = model says most likely to fall)
pool decline rate 0.5131
        decline_rate     pages  vs_pool
decile                                 
1             0.8923 1365.5000   0.3792
2             0.8089 1364.9000   0.2958
3             0.6455 1365.0000   0.1324
4             0.6167 1364.8000   0.1036
5             0.5312 1365.0000   0.0181
6             0.4375 1364.9000  -0.0756
7             0.3675 1364.7000  -0.1456
8             0.2782 1365.1000  -0.2348
9             0.2270 1364.8000  -0.2861
10            0.2081 1365.3000  -0.3050

monotonic across all ten deciles: True
top decile 0.8923 | bottom decile 0.2081 | spread 0.6842
first decile no better than the pool base rate: 6

does the ranking work at every traffic volume?
quartile       pages         impr_90d range    base     lift   ceiling


Q1 lowest     11,291         2 -       288  0.5533  +0.1349    30.2%


Q2            11,261       289 -     2,371  0.5412  +0.2443    53.2%


Q3            11,270     2,372 -     7,513  0.5003  +0.1681    33.6%


Q4 highest    11,273     7,515 -   797,764  0.4576  +0.3140    57.9%

pages already at zero: a different decision, not a queue position
gated pool reaching zero in the outcome window: 3,026 of 45,095 (6.71%)
of the top 100 per client:                      416 of 2,147 (19.38%)
median impr_90d of a dead queued page: 16 | of a live one: 896


**Verdict: the ordering degrades gracefully rather than falling off a cliff, which is the best news in
this notebook — but a fifth of the queue is pages that are already dead, and they need a different
decision entirely.**

**The ranking is monotonic across all ten deciles.** Actual decline rate falls **0.8923 → 0.8089 →
0.6455 → 0.6167 → 0.5312 → 0.4375 → 0.3675 → 0.2782 → 0.2270 → 0.2081**, a spread of **0.6842** against
a pool rate of **0.5131**, with no reversal anywhere. There is no point at which the list stops being
ordered, so a specialist working deeper into it is never getting noise — they are getting pages the
model is progressively more confident are *fine*.

**That sets the natural stopping point at decile 5.** Decile **6** is the first that is no better than
the pool base rate (**−0.0756** against it), and by decile 10 pages decline at **0.2081** — less than
half the pool rate. Working below the halfway mark is not merely low-yield, it is **worse than picking
at random from the same pool**, because the model is actively identifying those pages as healthy. If
budget remains after five deciles, the honest answer is that this queue has nothing left to say.

**Low-volume pages rank worst, but the pattern is not clean and should not be dressed up as one.**

| volume quartile | `impr_90d` | base rate | lift | share of ceiling |
|---|---|---|---|---|
| Q1 lowest | 2 – 288 | 0.5533 | +0.1349 | **30.2%** |
| Q2 | 289 – 2,371 | 0.5412 | +0.2443 | 53.2% |
| Q3 | 2,372 – 7,513 | 0.5003 | +0.1681 | 33.6% |
| Q4 highest | 7,515 – 797,764 | 0.4576 | +0.3140 | **57.9%** |

Q1 is the weakest and Q4 the strongest, which fits the noise argument — a page moving from 3
impressions to 1 is a 67% drop that means nothing. But **Q3 falls back to 33.6%** and breaks any tidy
monotonic story. Each stratum is scored on a fraction of the clients, so these estimates are noisier
than the pooled figures elsewhere, and the honest claim is narrow: **the bottom quartile is the
weakest place to spend review time**, not that skill rises smoothly with traffic.

**A fifth of the queue is already gone.** **6.71%** of the gated pool reaches zero in the outcome
window, but **19.38%** of the top 100 per client does — nearly three times over-represented. Their
median `impr_90d` is **16**, against **896** for the live pages beside them. The model is doing what it
was asked to: a page falling to zero is the largest possible decline, so it sorts to the top. **A page
at zero does not need a refresh, it needs a redirect, a merge, or a deletion** — and that decision is
irreversible in a way a content edit is not.

Note this is the same rule twice. The dead pages sit deep inside Q1, so **one volume floor removes the
weakest-ranked stratum and most of the already-dead pages together.**

### The no-go list

| Do not act on | Because |
|---|---|
| Anything past **decile 5** | at or below the pool base rate — the model is saying these pages are healthy |
| Pages below a **volume floor** | Q1 captures only **30.2%** of its ceiling and holds most of the dead pages |
| Pages already at **zero** | **19.38%** of the queue; they need a redirect or merge decision, not a refresh |
| Ranked queues for **saturated clients** | **13 of 30** clients receive their whole eligible list at K = 100 — that is the gate, not a ranking |
| The **predicted magnitude**, ever | ML-08 measured **R² 0.0026**; only the ordering is supported |

### What a person must check before acting

**Is the drop real, or is it instrumentation?** ML-04's sweep found single page-days carrying
**64,750** pageviews against a p99 of five, and **8.9 hours** of engagement on one page in one day.
Those are tabs left open, bots, or faults. A page whose "decline" is the disappearance of one such
spike did not lose readers.

**Read the reason code as an input, not a diagnosis.** **82.3%** of top-of-queue pages carry "running
below its own 90-day average", which restates what the model was fed rather than explaining anything
about the page. It says where to look, never why.

**Check GA4 coverage before trusting any engagement context.** No client tracks all of its content;
ML-09 measured the median at **46.2%**, and six clients have no GA4 at all.

### Never automated

The model does not edit, delete, redirect or merge anything, and nothing in this project supports
letting it. Deletion and redirection are irreversible; the model's own ranking is worth **+0.0787**
over a one-line filter on the segment where most of those candidates sit. That is not a margin to
hand an irreversible action to.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

ML-09 §8 answered the obvious version of this and got an unobvious answer: **staleness is not the
problem.** Holding the test month fixed and varying only the training date, a 92-day-old model scored
**0.1238** where a 31-day-old one scored **0.1242** — a gap of **0.0009**. Retraining frequency is not
the lever it looks like.

What *did* move was the pool. The decline rate ran **0.3169 → 0.5131 → 0.6146 → 0.7869** across four
monthly decision points, and since a queue cannot beat random by more than `1 − base_rate`, that
single number sets the ceiling on everything this playbook reports.

**That walk was spaced monthly, and FlyRank rebuilds weekly.** ML-09 flagged its own spacing as
answering the cadence the project assumed rather than the one it has. This section rebuilds it at the
real cadence: the warehouse allows decision points from 2026-03-01 to 2026-06-01, which at weekly
spacing is fourteen points and thirteen forward steps, against four and three before. The staleness
result rested on two comparisons; monitoring thresholds should not.

Each step trains on one week's cohort and scores the next, never backwards. The overlap caveat from
ML-09 applies more strongly here, not less: consecutive 90-day histories now share 83 of 90 days, so
these steps are heavily dependent and cannot establish a trend. What they can do is measure
**week-to-week variation**, which is exactly what a monitoring threshold needs.

In [6]:
WEEKLY = [d.strftime("%Y-%m-%d")
          for d in pd.date_range("2026-03-01", "2026-06-01", freq="7D")]
print(f"{len(WEEKLY)} weekly decision points | {len(WEEKLY) - 1} forward steps")
print(f"first {WEEKLY[0]} | last {WEEKLY[-1]}")

wk = {}
for n, d in enumerate(WEEKLY, 1):
    c = build_cohort(d)
    wk[d] = c[c["in_gate"]].dropna(subset=FULL).copy()
    print(f"  [{n:>2}/{len(WEEKLY)}] {d}  {len(wk[d]):>6,} pages | "
          f"{wk[d]['client_hash_id'].nunique():>2} clients | "
          f"decline rate {wk[d]['declined'].mean():.4f}")

rows = []
for a, b in zip(WEEKLY, WEEKLY[1:]):
    tr, te = wk[a], wk[b]
    sc = StandardScaler().fit(tr[LEAN])
    m = Ridge(alpha=1.0).fit(sc.transform(tr[LEAN]), tr["target"])
    rng = np.random.default_rng(0)
    t = te.assign(pred=m.predict(sc.transform(te[LEAN])), rand=rng.random(len(te)))
    pm, mm, _ = queue_metrics(t, "pred", 100, True)
    pr, mr, _ = queue_metrics(t, "rand", 100, False)
    br = te["declined"].mean()
    rows.append({"train": a, "test": b, "base_rate": br,
                 "pooled_lift": pm - pr, "macro_lift": mm - mr,
                 "pct_ceiling": (pm - pr) / (1 - br)})

wf = pd.DataFrame(rows)
print("\nweekly walk-forward, K = 100, lift over random at the same K")
print(wf.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print(f"\nsteps beating random: {(wf['pooled_lift'] > 0).sum()} of {len(wf)}")
print(f"pooled lift  mean {wf['pooled_lift'].mean():.4f} | "
      f"sd {wf['pooled_lift'].std():.4f} | "
      f"range {wf['pooled_lift'].min():.4f} - {wf['pooled_lift'].max():.4f}")
print(f"share of ceiling mean {wf['pct_ceiling'].mean():.1%} | "
      f"sd {wf['pct_ceiling'].std():.1%} | "
      f"range {wf['pct_ceiling'].min():.1%} - {wf['pct_ceiling'].max():.1%}")

# What actually moves the reported number: the pool, or the model?
base = pd.Series([wk[d]["declined"].mean() for d in WEEKLY], index=WEEKLY)
step = base.diff().dropna()
print(f"\nbase rate {base.min():.4f} - {base.max():.4f} across the fourteen points")
print(f"week-over-week change: mean {step.mean():+.4f} | sd {step.std():.4f} | "
      f"largest single move {step.abs().max():.4f}")
print(f"weeks where it moved more than 0.05: {int((step.abs() > 0.05).sum())} of {len(step)}")

r_lift_base = wf["pooled_lift"].corr(wf["base_rate"])
r_ceil_base = wf["pct_ceiling"].corr(wf["base_rate"])
print(f"\ncorrelation with the test week's base rate:")
print(f"  raw pooled lift   {r_lift_base:+.4f}   <- what a dashboard would plot")
print(f"  share of ceiling  {r_ceil_base:+.4f}   <- what actually reflects the model")

# A monitoring threshold has to be wider than ordinary week-to-week noise.
lo = wf["pct_ceiling"].mean() - 2 * wf["pct_ceiling"].std()
print(f"\nproposed alert floor: share of ceiling below {lo:.1%} "
      f"(mean minus two standard deviations of observed weekly variation)")
print(f"weeks in this sample that would have fired: "
      f"{int((wf['pct_ceiling'] < lo).sum())} of {len(wf)}")

14 weekly decision points | 13 forward steps
first 2026-03-01 | last 2026-05-31


  [ 1/14] 2026-03-01  36,272 pages | 31 clients | decline rate 0.3169


  [ 2/14] 2026-03-08  37,015 pages | 31 clients | decline rate 0.3472


  [ 3/14] 2026-03-15  37,968 pages | 32 clients | decline rate 0.3769


  [ 4/14] 2026-03-22  42,582 pages | 34 clients | decline rate 0.4033


...

  [11/14] 2026-05-10  38,769 pages | 32 clients | decline rate 0.6429


  [12/14] 2026-05-17  36,261 pages | 32 clients | decline rate 0.6844


  [13/14] 2026-05-24  37,611 pages | 31 clients | decline rate 0.7478


  [14/14] 2026-05-31  38,725 pages | 31 clients | decline rate 0.7873



weekly walk-forward, K = 100, lift over random at the same K
     train       test  base_rate  pooled_lift  macro_lift  pct_ceiling
2026-03-01 2026-03-08     0.3472       0.2274      0.1665       0.3484
2026-03-08 2026-03-15     0.3769       0.2559      0.1841       0.4106
2026-03-15 2026-03-22     0.4033       0.2404      0.1641       0.4029
2026-03-22 2026-03-29     0.4915       0.2478      0.1780       0.4873
2026-03-29 2026-04-05     0.5544       0.2277      0.1581       0.5110
2026-04-05 2026-04-12     0.6027       0.1795      0.1290       0.4518
2026-04-12 2026-04-19     0.6318       0.1679      0.1163       0.4561
2026-04-19 2026-04-26     0.6279       0.1615      0.1197       0.4340
2026-04-26 2026-05-03     0.6103       0.1973      0.1472       0.5063
2026-05-03 2026-05-10     0.6429       0.1841      0.1372       0.5154
2026-05-10 2026-05-17     0.6844       0.1373      0.1000       0.4351
2026-05-17 2026-05-24     0.7478       0.1140      0.0829       0.4519
2026-05-24 2026

### 4b. Before setting a trigger on the base rate, check what the base rate is measuring

The walk above shows the decline rate climbing from **0.3169** to **0.7873**, and ML-09 §8 reported the
same movement across months and called the pool non-stationary. Before any alert is built on that
number, it is worth asking what moved.

The target compares a page's **future 30 days** against its **baseline**, days 30–90 before the
decision. Both windows sit on the same panel. If the panel's own impression volume rises and then
falls — and a check of the daily totals says it does, with impressions per page-day peaking in
mid-April and falling by roughly a third into June while the row count holds — then *every* page's
future is compared against a baseline drawn from a different part of that curve. A decision point
early in the panel compares a rising future against a low baseline; a late one compares a falling
future against a high baseline.

That would produce exactly the monotonic rise observed, without any client's content getting worse.
The test below computes, for each decision point, the ratio of mean impressions-per-page-day in its
**future** window to the same quantity in its **baseline** window — a number about the panel that knows
nothing about any individual page — and asks how much of the decline rate it explains.

In [7]:
panel = con.sql(f"""
    SELECT report_date, SUM(gsc_impressions) AS impr, COUNT(*) AS rows_
    FROM {REL} GROUP BY 1 ORDER BY 1""").df()
panel["ipr"] = panel["impr"] / panel["rows_"]
panel.index = pd.to_datetime(panel["report_date"])


def panel_mean(d0, d1):
    """Mean impressions per page-day over [d0, d1) -- a panel-level quantity."""
    return float(panel.loc[(panel.index >= d0) & (panel.index < d1), "ipr"].mean())


prows = []
for d in WEEKLY:
    D = pd.Timestamp(d)
    fut = panel_mean(D, D + pd.Timedelta(days=30))
    bas = panel_mean(D - pd.Timedelta(days=90), D - pd.Timedelta(days=30))
    prows.append({"decision": d, "panel_baseline": bas, "panel_future": fut,
                  "panel_ratio": fut / bas, "base_rate": wk[d]["declined"].mean()})

pt = pd.DataFrame(prows)
print("the panel's own trajectory beside the decline rate it produces")
print(pt.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

r = pt["panel_ratio"].corr(pt["base_rate"])
print(f"\ncorrelation(panel future/baseline ratio, decline rate) = {r:+.4f}")
print(f"R-squared: {r ** 2:.4f} -- the share of the decline rate's movement that a "
      f"panel-level number explains")
print(f"\npanel ratio falls {pt['panel_ratio'].iloc[0]:.4f} -> {pt['panel_ratio'].iloc[-1]:.4f}"
      f"  while the decline rate rises {pt['base_rate'].iloc[0]:.4f} -> {pt['base_rate'].iloc[-1]:.4f}")
print(f"panel ratio crosses 1.0 (future stops exceeding baseline) between "
      f"{pt.loc[pt['panel_ratio'] >= 1, 'decision'].iloc[-1]} and "
      f"{pt.loc[pt['panel_ratio'] < 1, 'decision'].iloc[0]}")
print(f"the decline rate crosses 0.5 between "
      f"{pt.loc[pt['base_rate'] < 0.5, 'decision'].iloc[-1]} and "
      f"{pt.loc[pt['base_rate'] >= 0.5, 'decision'].iloc[0]}")

# Does the ranking survive the confound? The panel trend shifts every page in a
# cohort together, so it should move the base rate without touching the ordering.
print(f"\nfor contrast, the model's share of ceiling across the same weeks:")
print(f"  mean {wf['pct_ceiling'].mean():.1%} | sd {wf['pct_ceiling'].std():.1%} | "
      f"correlation with the panel ratio "
      f"{wf['pct_ceiling'].corr(pt['panel_ratio'].iloc[1:].reset_index(drop=True)):+.4f}")

the panel's own trajectory beside the decline rate it produces
  decision  panel_baseline  panel_future  panel_ratio  base_rate
2026-03-01         16.2783       28.3892       1.7440     0.3169
2026-03-08         17.1327       28.5973       1.6692     0.3472
2026-03-15         18.0449       28.8884       1.6009     0.3769
2026-03-22         19.2637       30.2686       1.5713     0.4033
2026-03-29         20.8179       28.8175       1.3843     0.4915
2026-04-05         22.0058       27.3617       1.2434     0.5544
2026-04-12         23.3984       25.8877       1.1064     0.6027
2026-04-19         24.3881       24.1915       0.9919     0.6318
2026-04-26         25.5349       22.9778       0.8999     0.6279
2026-05-03         26.7598       22.7956       0.8519     0.6103
2026-05-10         27.7454       22.0277       0.7939     0.6429
2026-05-17         28.3950       20.9746       0.7387     0.6844
2026-05-24         28.8655       19.5435       0.6771     0.7478
2026-05-31         28.2966 

**Verdict: the base rate is not a signal about clients, and must never be a trigger. It is the panel's
own impression curve, and it explains almost all of it.**

**A number that knows nothing about any page explains 95.7% of the decline rate.** The ratio of
panel-wide impressions-per-page-day in the future window to the same quantity in the baseline window
correlates with the decline rate at **−0.9784**, R² **0.9573**. As that ratio falls **1.7440 → 0.6574**,
the decline rate rises **0.3169 → 0.7873**.

The mechanism is arithmetic, not behaviour. Panel impressions per page-day climb through the spring
and fall through June, and the target compares a page's future against a baseline drawn from 30–90
days earlier. An early decision point measures a **rising** future against a **low** baseline —
panel baseline **16.2783** against future **28.3892** — and few pages can look declined. A late one
measures a **falling** future against a **high** baseline — **28.2966** against **18.6018** — and most
must. The panel ratio crosses 1.0 between **2026-04-12** and **2026-04-19**; the decline rate crosses
0.5 a fortnight earlier, between **2026-03-29** and **2026-04-05**. Same curve, small offset.

> ⚠️ **This corrects ML-09 §8 and the capstone.** Both state that the pool this model would be pointed
> at is **not stationary**, and treat the base rate moving from 0.3169 to 0.7869 as a fact about the
> clients that every grouped split was blind to. The *observation* stands and the blindness stands.
> The *attribution* does not: what moves is the panel, and a decline rate read at a single decision
> point says more about where that date sits on the panel's curve than about anyone's content.
> Whether the curve itself is genuine seasonality, the panel growing from 43 to 66 clients, or an
> artefact of how the warehouse was assembled is **not something this data can settle** — but the
> label is measuring it either way.

**The ranking is untouched, and that is the reason to trust it.** A panel-wide shift moves every page
in a cohort together, so it changes the base rate without disturbing the order. **13 of 13** forward
steps beat random. Share of ceiling holds at **45.8%** with an sd of **5.4%** across the same fourteen
weeks in which the base rate more than doubled, and its correlation with the panel ratio is
**−0.6208** — present, far weaker, and in a stable band. **Level moves; ordering does not.**

### What to monitor, and what to ignore

**Do not alert on raw `P@K` or raw lift.** Across these thirteen steps pooled lift fell **0.2559 →
0.1140**, a drop of more than half, and correlates **−0.9135** with the test week's base rate. A
dashboard plotting the obvious number would have shown a model in freefall and triggered a retrain of
something that never degraded. **Do not alert on the base rate either** — that is the panel, as above.

**Alert on share of ceiling**, which correlates **+0.6575** with the base rate rather than −0.9135, and
is the only reported quantity that reflects the model. On observed weekly variation, mean minus two
standard deviations puts the floor at **35.0%**; **1 of 13** weeks would have fired. That count is
in-sample — the threshold was derived from the same thirteen steps it is tested on — so treat it as a
starting point to be re-estimated once real weeks accumulate, not as a validated false-positive rate.

**Retrain rarely.** ML-09 §8c measured a 92-day-old model at **0.1238** against a 31-day-old one at
**0.1242**. Staleness is not what moves this. Weekly rebuilds of the *queue* are what FlyRank does and
what the cadence requires; weekly refits of the *model* would buy **0.0009** and cost a weekly
opportunity to introduce a fault.

**Watch the pool, not the score.** Week-over-week the base rate moves **+0.0362** on average with an sd
of **0.0275**, and **3 of 13** weeks moved more than 0.05. Those moves are the panel and are expected.
What is worth an alert is composition: a client whose gated pool crosses its own budget, since §1
showed **13 of 30** clients receive their whole list at K = 100 and stop being ranked at all.

**Stated limit.** Consecutive 90-day histories share 83 of 90 days, so these thirteen steps are heavily
dependent — they measure week-to-week variation, which is what a threshold needs, and cannot establish
a trend. The panel-confound test is a correlation across fourteen overlapping points, not a controlled
comparison; it shows the decline rate tracks the panel almost exactly, not that nothing else moves it.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to `work/outputs/`.*

Two things this export has to get right, and both are checked in code rather than promised.

**The queue must not contain the answer.** Every metric in this notebook is computed against `target`,
`declined` and `future_impr` — all three describe what happened *after* the decision point. A queue
handed to a specialist on the decision date cannot contain any of them, or it is not a queue, it is a
mark sheet. The export asserts their absence rather than relying on having remembered.

**Nothing leaves this repo that identifies anyone.** `DATA_USE.md` permits pseudonymous hashes; it
does not permit URLs, client names or raw queries, and none of those are in the frame anyway. The
export asserts that every non-numeric column is either a hash or one of this notebook's own generated
strings.

`work/**/*.csv` is gitignored and CI hard-fails on any committed CSV outside two allowlisted paths, so
these files stay local by construction. They are **generated artefacts, never a source of figures** —
every number in this project has to come from a cell that ran, and a CSV sitting in a folder is
exactly the stale second source of truth `check_claims.py` exists to catch.

In [8]:
from pathlib import Path
import json

OUTDIR = Path("../outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

assert abs(r) > 0.9, f"r is not the panel correlation any more: {r}"
FLOOR = float(ev["impr_90d"].quantile(0.25))     # section 3's weakest volume quartile
export = queue.copy()
export["segment"] = np.where(export["peak_ratio"] < 1,
                             "filter_output_already_below_average",
                             "ranked_not_yet_falling")
export["below_volume_floor"] = export["impr_90d"] < FLOOR
export = export[["client_hash_id", "content_hash_id", "rank_in_client", "predicted_change",
                 "reason_1", "reason_2", "segment", "below_volume_floor",
                 "impr_90d", "avg_position", "prior_trend", "peak_ratio"]]

# --- guard 1: the outcome must not be in a forward-looking artefact ---------------
FUTURE = {"target", "declined", "future_impr", "future_daily", "fut_7", "fut_14",
          "fut_30", "fut_60", "fut_90"}
leaked = FUTURE & set(export.columns)
assert not leaked, f"outcome columns in the queue export: {sorted(leaked)}"
print(f"guard 1 -- no outcome column in the export: PASS ({len(FUTURE)} checked)")

# --- guard 2: nothing identifying leaves the repo --------------------------------
import pandas.api.types as ptypes

GENERATED = {"reason_1", "reason_2", "segment"}
ID_COLS = {"client_hash_id", "content_hash_id"}
# pandas 3 gives string columns dtype "str", not "object". The first version of this
# test asked `dtype == object`, matched nothing, and printed PASS having inspected
# zero columns -- a URL column would have sailed through it. Anything not numeric and
# not boolean is text, and a guard that cannot find its subject must fail, not pass.
text_cols = [c for c in export.columns
             if not (ptypes.is_numeric_dtype(export[c]) or ptypes.is_bool_dtype(export[c]))]
assert text_cols, "no text columns detected -- the dtype test is broken, not the data"
unexpected = [c for c in text_cols if c not in GENERATED | ID_COLS]
assert not unexpected, f"unexpected text columns: {unexpected}"
# The pseudonym format is a prefix plus 16 lowercase hex, e.g. client_d211cb07b9059bab.
# The first version of this assertion demanded bare hex and fired on the real data --
# the guard was right and the pattern was wrong. Matching the prefix and the exact
# width is stricter than what it replaced, not a loosening to make the run pass.
HASH_RE = r"(?:client|content)_[0-9a-f]{16}"
hashes = pd.concat([export["client_hash_id"], export["content_hash_id"]]).astype(str)
bad = hashes[~hashes.str.fullmatch(HASH_RE)]
assert bad.empty, f"not a pseudonymous hash: {bad.unique()[:3].tolist()}"
assert not hashes.str.contains(r"https?://|www\.|\.com", case=False).any(), "URL-like value found"
print(f"guard 2 -- every text column is a hash or generated: PASS "
      f"({len(text_cols)} text columns, {hashes.nunique():,} distinct ids checked)")

qpath = OUTDIR / "ml10_review_queue.csv"
export.to_csv(qpath, index=False)

summary = {
    "decision_date": D1,
    "gated_pool_pages": int(len(ev)),
    "clients": int(ev["client_hash_id"].nunique()),
    "pool_decline_rate": round(float(ev["declined"].mean()), 4),
    "features": LEAN,
    "queue_rows": int(len(export)),
    "volume_floor_impr_90d": round(FLOOR, 1),
    "segment_counts": {k: int(v) for k, v in export["segment"].value_counts().items()},
    "weekly_walk": {
        "steps": int(len(wf)),
        "steps_beating_random": int((wf["pooled_lift"] > 0).sum()),
        "pooled_lift_mean": round(float(wf["pooled_lift"].mean()), 4),
        "pct_ceiling_mean": round(float(wf["pct_ceiling"].mean()), 4),
        "pct_ceiling_sd": round(float(wf["pct_ceiling"].std()), 4),
        "alert_floor_pct_ceiling": round(float(lo), 4),
    },
    "panel_confound_r2": round(float(r ** 2), 4),
    "generated_by": "work/notebooks/w07_action_playbook.ipynb",
    "warning": "generated artefact -- never cite figures from this file, recompute them",
}
spath = OUTDIR / "ml10_summary.json"
spath.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"\nwrote {qpath.name:<26} {len(export):>6,} rows x {export.shape[1]} cols "
      f"({qpath.stat().st_size / 1024:,.0f} KB)")
print(f"wrote {spath.name:<26} {len(summary)} keys "
      f"({spath.stat().st_size / 1024:,.1f} KB)")
print(f"\nboth under work/outputs/, which is gitignored for csv and never committed")
print(f"\nqueue composition:")
for seg, n in export["segment"].value_counts().items():
    print(f"  {seg:<38} {n:>5,} ({n / len(export):.1%})")
print(f"  below the volume floor (impr_90d < {FLOOR:,.0f})   "
      f"{int(export['below_volume_floor'].sum()):>5,} "
      f"({export['below_volume_floor'].mean():.1%})")

guard 1 -- no outcome column in the export: PASS (9 checked)


guard 2 -- every text column is a hash or generated: PASS (5 text columns, 45,125 distinct ids checked)



wrote ml10_review_queue.csv      45,095 rows x 12 cols (10,686 KB)
wrote ml10_summary.json          12 keys (0.8 KB)

both under work/outputs/, which is gitignored for csv and never committed

queue composition:
  ranked_not_yet_falling                 31,685 (70.3%)
  filter_output_already_below_average    13,410 (29.7%)
  below the volume floor (impr_90d < 288)   11,272 (25.0%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — §5 asserts this in code rather than
      claiming it, and the export carries only pseudonymous hashes and generated strings
- [x] My claims use careful words: observed, measured, directional, decision-support

## Revision log

Three things this notebook set out to show, and then disproved.

| stated here | what the test showed |
|---|---|
| §1: "the queue describes the present rather than forecasting the future" — inferred from **82.3%** of top slots carrying one reason code | queue *composition* cannot settle this, and the reasoning was circular: `peak_ratio` drives the score, so of course it dominates the reasons. §2 split the pool on that exact condition and found comparable skill on both sides — **33.4%** of ceiling on already-falling pages, **36.9%** on pages that had not moved |
| §2: the model is three times weaker on already-falling pages (**+0.0787** against **+0.2186**) | mostly ceiling. Those segments decline at **0.7641** and **0.4069**, so headroom differs by more than two to one. Normalised, the gap nearly closes |
| §4: ML-09's finding that the pool "is not stationary" across decision points | the *panel* is not stationary. A panel-level ratio explains **95.7%** of the decline rate's movement (R² **0.9573**), and it knows nothing about any individual page. ML-09 §8 and the capstone are corrected in place |

**What survived every attack.** The ranking. It is monotonic across all ten deciles, beats random on
**13 of 13** forward weekly steps, holds **45.8%** of its available ceiling with an sd of **5.4%**
while the base rate more than doubles, and is equally skilful at outcome windows from **7** to **90**
days. Every finding that broke was about a *level*; nothing that broke was about an *order*.

**The one number that matters operationally.** A page already below its own 90-day average declines
**76.41%** of the time — a one-line filter, no model required. The ranking adds **+0.0787** there and
**+0.2186** where a filter would leave you at **40.69%**. The model earns its place on pages that have
not started falling, and the global queue sends most of its slots to the other segment.